In [1]:
import ast
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import root_mean_squared_error

In [2]:
# ----------------------------
# Configuration
# ----------------------------

PATH = "../data/raw/listings.csv"
RANDOM_STATE = 42
TEST_SIZE = 0.20
CV_FOLDS = 5

# Keep this list synchronized with your experimentation notebook.
COLS_TO_DROP = [
    # identifiers / urls / metadata
    "id", "listing_url", "scrape_id", "source", "picture_url",
    "host_id", "host_url", "host_thumbnail_url", "host_picture_url",
    "calendar_updated", "calendar_last_scraped",

    # leakage
    "estimated_revenue_l365d", "estimated_occupancy_l365d",

    # text fields (no NLP)
    "name", "description", "neighborhood_overview", "host_about",

    # redundant text versions
    "bathrooms_text", "host_name", "host_verifications",

    # review score redundancy
    "review_scores_accuracy", "review_scores_checkin",
    "review_scores_communication", "review_scores_value",

    # availability redundancy
    "availability_60", "availability_eoy",

    # review activity redundancy
    "number_of_reviews_ltm", "number_of_reviews_l30d",
    "number_of_reviews_ly",

    # derived night statistics
    "minimum_minimum_nights", "maximum_minimum_nights",
    "minimum_maximum_nights", "maximum_maximum_nights",
    "minimum_nights_avg_ntm", "maximum_nights_avg_ntm",

    # categorical removal from EDA
    "first_review", "last_review", "license",
    "host_location", "host_neighbourhood", "neighbourhood",
    "neighbourhood_group_cleansed", "has_availability",
    "host_identity_verified",

    # weak categorical predictor
    "host_response_time", "host_response_rate",
]

CLIP_COLS = [
    "beds",
    "bathrooms",
    "bedrooms",
    "minimum_nights",
    "maximum_nights",
    "host_acceptance_rate_num",
]

LOG_COLS = [
    "minimum_nights",
    "accommodates",
    "maximum_nights",
    "number_of_reviews",
    "reviews_per_month",
]

TOP_K = 10
AMENITIES_MIN_SHARE = 0.05

# If you want Ridge to use only engineered distance features instead of raw coordinates,
# leave this as True. Set it to False if you want to keep latitude/longitude too.
DROP_LAT_LON = True

DISTANCE_POINTS = {
    "distance_to_bilbao":   (43.2630, -2.9350),
    "distance_to_donostia": (43.3183, -1.9812),
    "distance_to_vitoria":  (42.8467, -2.6726),
    "distance_to_coast":    (43.3623, -3.0136),
}

In [3]:
# ----------------------------
# Helper functions
# ----------------------------

def clean_price(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .replace("", np.nan)
        .astype(float)
    )

def clean_percentage(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace("%", "", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce") / 100

def parse_amenities(value):
    if pd.isna(value) or value == "":
        return []
    if isinstance(value, list):
        parsed = value
    else:
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
    return [str(a).strip().strip('"') for a in parsed if str(a).strip()]

def safe_col(name: str) -> str:
    safe_name = (
        name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace("&", "and")
        .replace(".", "")
    )
    return "".join(ch for ch in safe_name if ch.isalnum() or ch == "_").strip("_")

In [4]:
# ----------------------------
# Custom transformers
# ----------------------------

class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        return X.drop(columns=self.columns, errors="ignore")


class AmenitiesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, amenities_col="amenities", min_share=0.05):
        self.amenities_col = amenities_col
        self.min_share = min_share
        self.selected_amenities_ = []
        self.amenity_column_map_ = {}

    def fit(self, X, y=None):
        X = X.copy()
        if self.amenities_col not in X.columns:
            self.selected_amenities_ = []
            return self

        amenities_lists = X[self.amenities_col].apply(parse_amenities)
        amenities_counts = pd.Series(
            [amenity for sublist in amenities_lists for amenity in sublist]
        ).value_counts()

        min_count = max(1, int(len(X) * self.min_share))
        self.selected_amenities_ = amenities_counts[amenities_counts >= min_count].index.tolist()

        used_cols = set()
        self.amenity_column_map_ = {}
        for amenity in self.selected_amenities_:
            base_col = f"has_{safe_col(amenity)}"
            col = base_col
            suffix = 2
            while col in used_cols:
                col = f"{base_col}_{suffix}"
                suffix += 1
            used_cols.add(col)
            self.amenity_column_map_[amenity] = col
        return self

    def transform(self, X):
        X = X.copy()
        if self.amenities_col not in X.columns:
            return X

        amenities_lists = X[self.amenities_col].apply(parse_amenities)
        amenities_sets = amenities_lists.apply(set)

        amenity_data = {
            col_name: amenities_sets.apply(lambda s, a=amenity: int(a in s))
            for amenity, col_name in self.amenity_column_map_.items()
        }

        amenity_df = pd.DataFrame(amenity_data, index=X.index)
        amenity_count = pd.Series(amenities_lists.apply(len), name="amenity_count", index=X.index)
        X = X.drop(columns=[self.amenities_col], errors="ignore")
        X = pd.concat([X, amenity_count, amenity_df], axis=1)
        return X


class HostExperienceTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, host_since_col="host_since", reference_col="last_scraped"):
        self.host_since_col = host_since_col
        self.reference_col = reference_col
        self.reference_date_ = None

    def fit(self, X, y=None):
        X = X.copy()
        ref = pd.to_datetime(X[self.reference_col], errors="coerce")
        self.reference_date_ = ref.max()
        return self

    def transform(self, X):
        X = X.copy()
        X[self.reference_col] = pd.to_datetime(X[self.reference_col], errors="coerce")
        X[self.host_since_col] = pd.to_datetime(X[self.host_since_col], errors="coerce")

        X["host_experience_days"] = (
            self.reference_date_ - X[self.host_since_col]
        ).dt.days

        return X.drop(columns=[self.host_since_col, self.reference_col], errors="ignore")


class DistanceFeaturesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, lat_col="latitude", lon_col="longitude", points=None, drop_lat_lon=True):
        self.lat_col = lat_col
        self.lon_col = lon_col
        self.points = points if points is not None else {}
        self.drop_lat_lon = drop_lat_lon

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        lat = pd.to_numeric(X[self.lat_col], errors="coerce")
        lon = pd.to_numeric(X[self.lon_col], errors="coerce")

        for feature_name, (ref_lat, ref_lon) in self.points.items():
            X[feature_name] = np.sqrt((lat - ref_lat) ** 2 + (lon - ref_lon) ** 2)

        if self.drop_lat_lon:
            X = X.drop(columns=[self.lat_col, self.lon_col], errors="ignore")

        return X


class NumericClipper(BaseEstimator, TransformerMixin):
    def __init__(self, columns, lower_q=0.01, upper_q=0.99):
        self.columns = columns
        self.lower_q = lower_q
        self.upper_q = upper_q
        self.bounds_ = {}

    def fit(self, X, y=None):
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                series = pd.to_numeric(X[col], errors="coerce")
                self.bounds_[col] = (
                    series.quantile(self.lower_q),
                    series.quantile(self.upper_q)
                )
        return self

    def transform(self, X):
        X = X.copy()
        for col, (lower, upper) in self.bounds_.items():
            X[col] = pd.to_numeric(X[col], errors="coerce").clip(lower, upper)
        return X


class NumericLog1pTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.columns:
            if col in X.columns:
                X[col] = np.log1p(pd.to_numeric(X[col], errors="coerce"))
        return X


class BinaryMapTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column, mapping):
        self.column = column
        self.mapping = mapping

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.column in X.columns:
            X[self.column] = X[self.column].map(self.mapping)
        return X


class TopKCategoryTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column, new_column=None, top_k=10, other_label="Other", drop_original=True):
        self.column = column
        self.new_column = new_column if new_column is not None else f"{column}_clean"
        self.top_k = top_k
        self.other_label = other_label
        self.drop_original = drop_original
        self.top_categories_ = None

    def fit(self, X, y=None):
        X = X.copy()
        counts = X[self.column].value_counts(dropna=True)
        self.top_categories_ = counts.nlargest(self.top_k).index.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        X[self.new_column] = X[self.column].where(
            X[self.column].isin(self.top_categories_),
            self.other_label
        )
        if self.drop_original:
            X = X.drop(columns=[self.column], errors="ignore")
        return X

In [5]:
# ----------------------------
# Load data and define target
# ----------------------------

df = pd.read_csv(PATH)

df["price"] = clean_price(df["price"])
df["host_response_rate"] = clean_percentage(df["host_response_rate"])
df["host_acceptance_rate"] = clean_percentage(df["host_acceptance_rate"])
df = df.rename(columns={"host_acceptance_rate": "host_acceptance_rate_num"})

df = df.dropna(subset=["price"]).copy()
df["log_price"] = np.log1p(df["price"])

df["price_bin"] = pd.qcut(df["price"], q=5, duplicates="drop")

df_train, df_test = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["price_bin"]
)

df_train = df_train.drop(columns=["price_bin"])
df_test = df_test.drop(columns=["price_bin"])

X_train = df_train.drop(columns=["price", "log_price"])
y_train = df_train["log_price"]

X_test = df_test.drop(columns=["price", "log_price"])
y_test = df_test["log_price"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4520, 78)
Test shape: (1131, 78)


In [6]:
# ----------------------------
# Shared feature-engineering pipeline
# ----------------------------

feature_engineering = Pipeline(
    steps=[
        ("drop_columns", DropColumnsTransformer(COLS_TO_DROP)),
        ("amenities", AmenitiesTransformer(min_share=AMENITIES_MIN_SHARE)),
        ("host_experience", HostExperienceTransformer()),
        ("distance_features", DistanceFeaturesTransformer(
            points=DISTANCE_POINTS,
            drop_lat_lon=DROP_LAT_LON
        )),
        ("clipper", NumericClipper(CLIP_COLS)),
        ("log1p", NumericLog1pTransformer(LOG_COLS)),
        ("instant_bookable_map", BinaryMapTransformer(
            column="instant_bookable",
            mapping={"t": 1, "f": 0}
        )),
        ("property_topk", TopKCategoryTransformer(
            column="property_type",
            new_column="property_type_clean",
            top_k=TOP_K
        )),
        ("neighbourhood_topk", TopKCategoryTransformer(
            column="neighbourhood_cleansed",
            new_column="neighbourhood_cleansed_clean",
            top_k=TOP_K
        )),
    ]
)

In [7]:
# ----------------------------
# Model-specific preprocessors
# ----------------------------

numeric_selector = make_column_selector(dtype_include=np.number)
categorical_selector = make_column_selector(dtype_exclude=np.number)

ridge_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_selector),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_selector),
    ],
    remainder="drop"
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), numeric_selector),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical_selector),
    ],
    remainder="drop"
)

In [8]:
# ----------------------------
# Candidate pipelines
# ----------------------------

ridge_pipe = Pipeline([
    ("features", feature_engineering),
    ("preprocess", ridge_preprocessor),
    ("model", Ridge())
])

rf_pipe = Pipeline([
    ("features", feature_engineering),
    ("preprocess", tree_preprocessor),
    ("model", RandomForestRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

hgb_pipe = Pipeline([
    ("features", feature_engineering),
    ("preprocess", tree_preprocessor),
    ("model", HistGradientBoostingRegressor(
        random_state=RANDOM_STATE
    ))
])

In [9]:
# ----------------------------
# Hyperparameter tuning on training data only
# ----------------------------

ridge_grid = {
    "model__alpha": np.logspace(-3, 3, 25),
}

rf_grid = {
    "model__n_estimators": [300, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
}

hgb_grid = {
    "model__learning_rate": [0.03, 0.05, 0.1],
    "model__max_iter": [200, 300],
    "model__max_depth": [None, 6, 10],
    "model__min_samples_leaf": [10, 20, 30],
    "model__l2_regularization": [0.0, 0.1, 1.0],
}

In [10]:
ridge_search = GridSearchCV(
    estimator=ridge_pipe,
    param_grid=ridge_grid,
    scoring="neg_root_mean_squared_error",
    cv=CV_FOLDS,
    n_jobs=-1,
    refit=True
)

rf_search = GridSearchCV(
    estimator=rf_pipe,
    param_grid=rf_grid,
    scoring="neg_root_mean_squared_error",
    cv=CV_FOLDS,
    n_jobs=-1,
    refit=True
)

hgb_search = GridSearchCV(
    estimator=hgb_pipe,
    param_grid=hgb_grid,
    scoring="neg_root_mean_squared_error",
    cv=CV_FOLDS,
    n_jobs=-1,
    refit=True
)

In [11]:
# WARNING:
# Running all three searches can take time.
# You can comment out the tree models temporarily if you want a faster pass.

ridge_search.fit(X_train, y_train)
print("Ridge best CV RMSE:", -ridge_search.best_score_)
print("Ridge best params:", ridge_search.best_params_)

Ridge best CV RMSE: 0.46901844026823375
Ridge best params: {'model__alpha': np.float64(1.7782794100389228)}


In [12]:
rf_search.fit(X_train, y_train)
print("RF best CV RMSE:", -rf_search.best_score_)
print("RF best params:", rf_search.best_params_)

RF best CV RMSE: 0.4048104207762081
RF best params: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 500}


In [13]:
hgb_search.fit(X_train, y_train)
print("HGB best CV RMSE:", -hgb_search.best_score_)
print("HGB best params:", hgb_search.best_params_)

HGB best CV RMSE: 0.3743005300740429
HGB best params: {'model__l2_regularization': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': None, 'model__max_iter': 300, 'model__min_samples_leaf': 10}


In [14]:
# ----------------------------
# Compare tuned models
# ----------------------------

results = pd.DataFrame({
    "model": ["ridge", "random_forest", "hist_gradient_boosting"],
    "cv_rmse_log": [
        -ridge_search.best_score_,
        -rf_search.best_score_,
        -hgb_search.best_score_,
    ]
}).sort_values("cv_rmse_log")

results

,model,cv_rmse_log
2,hist_gradient_boosting,0.374301
1,random_forest,0.404810
0,ridge,0.469018


In [15]:
# ----------------------------
# Select the best model
# ----------------------------

searches = {
    "ridge": ridge_search,
    "random_forest": rf_search,
    "hist_gradient_boosting": hgb_search,
}

best_model_name = results.iloc[0]["model"]
best_search = searches[best_model_name]
best_pipeline = best_search.best_estimator_

print("Selected model:", best_model_name)
print("Best CV RMSE (log scale):", -best_search.best_score_)

Selected model: hist_gradient_boosting
Best CV RMSE (log scale): 0.3743005300740429


In [16]:
# ----------------------------
# Final test evaluation
# ----------------------------

test_pred_log = best_pipeline.predict(X_test)

test_rmse_log = root_mean_squared_error(y_test, test_pred_log)

# Convert back to original euro scale
y_test_eur = np.expm1(y_test)
test_pred_eur = np.expm1(test_pred_log)

test_rmse_eur = root_mean_squared_error(y_test_eur, test_pred_eur)
mean_price = y_test_eur.mean()
rmse_pct_of_mean = 100 * test_rmse_eur / mean_price

print("Final test RMSE (log scale):", test_rmse_log)
print("Final test RMSE (€):", test_rmse_eur)
print("RMSE as % of mean test price:", rmse_pct_of_mean)

Final test RMSE (log scale): 0.361078675707378
Final test RMSE (€): 463.5767827801614
RMSE as % of mean test price: 217.41244970055294
